# Phase 2 distillation -- Step 4d LoRA training (Colab A100)

Trains a LoRA adapter on Qwen2.5-7B-Instruct over the 1000-example SFT corpus produced by step4a/b/c. Wallclock target ~30-60 min on A100, cost ~$0.60-1.20 on Colab Pay-as-you-go.

## Prerequisites

1. **Runtime**: Set Colab runtime to A100 GPU + High-RAM (Runtime -> Change runtime type).
2. **No data upload required.** The corpus (`train.jsonl`, `valid.jsonl`, `test.jsonl`) is committed to the `feature/phase2-distillation` branch under `experiments/phase2_distillation/sft_corpus_v1/mlx_split/` and lands automatically when Cell 3 clones the repo. If Colab disconnects mid-run, just re-open this notebook -- the data persists in the branch.

## Output

`adapter_smoke_v1.zip` -- the LoRA adapter (~80 MB) + `training_metadata.json`. Download via Cell 6 to integrate locally for Step 4d eval.

In [ ]:
# Cell 2: GPU sanity check
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f'cuda available: {torch.cuda.is_available()}')
print(f'device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Cell 3: Clone branch (corpus included) + verify
# BRANCH defaults to feature/phase2-distillation. After this PR
# merges to develop the corpus moves to develop, so set
# BRANCH='develop' (or any commit sha / tag) for re-runs.
BRANCH = 'feature/phase2-distillation'
%cd /content
!rm -rf merken  # idempotent: re-runnable on disconnect
!git clone --depth 1 --branch {BRANCH} https://github.com/stffns/merken.git
%cd /content/merken
import pathlib, json
data_dir = pathlib.Path('/content/merken/experiments/phase2_distillation/sft_corpus_v1/mlx_split')
for name in ('train.jsonl', 'valid.jsonl', 'test.jsonl'):
    p = data_dir / name
    assert p.exists(), f'missing {p} -- the {BRANCH} branch may not carry the corpus; check the branch name above'
    n = sum(1 for _ in p.open())
    print(f'  {p.name}: {n} rows')
first = json.loads(next(iter((data_dir/'train.jsonl').open())))
assert 'messages' in first, 'first row missing messages key'
print(f'  first row keys: {list(first.keys())}')

In [ ]:
# Cell 4: Install training dependencies
!pip install -q --upgrade transformers peft trl datasets accelerate bitsandbytes hf_transfer
import transformers, peft, trl, datasets
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'trl: {trl.__version__}')
print(f'datasets: {datasets.__version__}')

In [ ]:
# Cell 5: Run the trainer (~30-60 min on A100)
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
%cd /content/merken
!python -m experiments.phase2_distillation.train_qwen_lora \
    --data /content/merken/experiments/phase2_distillation/sft_corpus_v1/mlx_split \
    --output-dir /content/adapters/smoke_v1 \
    --base-model Qwen/Qwen2.5-7B-Instruct \
    --rank 16 --alpha 32 \
    --epochs 1 --batch-size 1 --grad-accum 4 \
    --max-seq-len 8192 \
    --lr 1e-4

In [ ]:
# Cell 6: Zip + download the adapter
import shutil, pathlib
src = pathlib.Path('/content/adapters/smoke_v1/adapter_final')
assert src.exists(), 'adapter not found -- training likely failed'
dst = pathlib.Path('/content/adapter_smoke_v1.zip')
shutil.make_archive(str(dst.with_suffix('')), 'zip', root_dir=str(src.parent), base_dir=src.name)
print(f'zip: {dst} ({dst.stat().st_size / 1e6:.1f} MB)')
from google.colab import files
files.download(str(dst))

## After download (local)

1. Unzip into `experiments/phase2_distillation/sft_corpus_v1/adapter_smoke_v1/`.
2. To run the trained student locally, two paths:
   - **Direct HF inference** (slow on Mac): `transformers + peft.PeftModel.from_pretrained` against the base + adapter. Works on MPS but slow.
   - **Fuse + convert to MLX** (fast on Mac): merge adapter into base via `peft.merge_and_unload`, save merged BF16, convert to MLX with `mlx_lm.convert --hf-path <merged> -q --q-bits 8`. Then load in LM Studio.
3. Step 4d eval: re-run `step3_zero_shot_smoke.py` against the trained-and-fused model. Compare per-shape reasoning length + truncation rate vs the zero-shot baseline. Gate: >=5pp on LoCoMo seed=44 N=201 (canonical eval, separate sweep).

## Resilience to disconnects

Cell 3 is idempotent (`rm -rf` first). On Colab disconnect, just re-open the notebook URL and re-run cells 2-5. The corpus is in the branch so no upload is needed; only the wallclock spent so far in Cell 5 is lost. To resume training mid-run, replace Cell 5 with a `--resume-from-checkpoint <output_dir>` invocation (the trainer's `save_strategy=epoch` saves checkpoints at each epoch end).

## Cost ledger

Training: ~$0.60-1.20 (Colab Pay-as-you-go A100, 30-60 min).
Smoke eval: $0 (local).
Total Step 4d: ~$1.